# RAG From Scratch: Leanring Breakdown

## Project Goal

The source notebooks are structured for learning: Each concept is explained by a markdown cell + code following it. The RAG model should preserve that relationship rather than indexing Markdown and code as unrelated fragments ( this helps mianing the relationship between the concept and the code implementation)

A user should be able to ask follow-ups such as:

- "What is calibration in the evaluation notebook?"
- "Show me the code that applies that idea."
- "Break that explanation down more simply."
- "How does that preprocessing choice affect Ridge regression?"

Cloud will retrieve relevant notebook evidence first, then use a local LLM to create a grounded answer with notebook citations.


## Design Principle

The core retrieval unit should be:

$$
\text{concept explanation} + \text{implementation example} + \text{source metadata}
$$

### General guideline for RAG systems:
- → parse documents
- → create chunks
- → index chunks
- → retrieve relevant evidence
- → build a grounded prompt
- → generate an answer
- → return sources

---


## Phase 1: Parse the Notebooks

- Use `nbformat` to read each `.ipynb` file in `foundations_and_models/`. 
- Extract only Markdown and code cells (preserving the notebook name, original cell position, and cell type) *Do not index execution counts, display output, images, warnings, or error tracebacks in the first version.*

    - Each extracted cell should become a structured record:

    ```python
    {
        "content": "...",
        "notebook": "preprocessing.ipynb",
        "cell_index": 12,
        "cell_type": "markdown"
    }
    ```

**Checkpoint:** inspect at least 10 records manually. Confirm the content is meaningful and the source metadata can identify where each record came from.




In [7]:
from pathlib import Path
import nbformat

# Find the repository's foundations_and_models folder from either the repo root
# or the modern_ml_reponsible_ai notebook directory.
working_dir = Path.cwd()
folder_candidates = [
    working_dir / "foundations_and_models",
    working_dir.parent / "foundations_and_models",
]
folder_path = next((path for path in folder_candidates if path.exists()), None)

if folder_path is None:
    raise FileNotFoundError(
        "Could not find the foundations_and_models folder from the current working directory."
    )

notebook_files = sorted(folder_path.glob("*.ipynb"))
print(f"Found {len(notebook_files)} notebook(s) in {folder_path}")

notebook_cells = []

for file_path in notebook_files:
    file_name = file_path.name
    print(f"Processing: {file_name}")

    try:
        notebook = nbformat.read(file_path, as_version=4)
    except Exception as error:
        print(f"Failed to read {file_name}: {error}")
        continue

    for cell_index, cell in enumerate(notebook.cells):
        if cell.cell_type not in {"markdown", "code"}:
            continue

        content = cell.source.strip()
        if not content:
            continue

        notebook_cells.append(
            {
                "content": content,
                "notebook": file_name,
                "cell_index": cell_index,
                "cell_type": cell.cell_type,
            }
        )

print(f"\nExtracted {len(notebook_cells)} Markdown and code cell(s).\n")

# Inspect a few records before moving on to paired chunking.
print("Showing the first 3 extracted notebook cells:")
for record in notebook_cells[:10]:
    print(
        f"{record['notebook']} | cell {record['cell_index']} | "
        f"{record['cell_type']} | {record['content'][:100]!r}"
    )

Found 4 notebook(s) in /Users/claudiafarkas/Development/crashcourses/crashcourses/foundations_and_models
Processing: model_evaluation_error_analysis.ipynb
Processing: preprocessing.ipynb
Processing: supervised_learning.ipynb
Processing: unsupervised_learning.ipynb

Extracted 93 Markdown and code cell(s).

Showing the first 3 extracted notebook cells:
model_evaluation_error_analysis.ipynb | cell 0 | markdown | '# Model Evaluation & Error Analysis\n\n### *Evaluation is how we decide whether a model is useful for '
model_evaluation_error_analysis.ipynb | cell 1 | markdown | "## 1. Evaluating a Regression Model\n\n### The goal\n\nWe want to predict a home's **price** from its **"
model_evaluation_error_analysis.ipynb | cell 2 | markdown | '### Step 1: Split the data into training and test homes\n\nThe linear regression model is the same one'
model_evaluation_error_analysis.ipynb | cell 3 | code | '# TODO: Load the housing data, define features and the price target, and create train/test sp

## Phase 2: Create Paired Chunks

Start with one cell per chunk as a baseline. Then add the notebook-aware strategy that fits this repository:

1. For every code cell, find the closest preceding meaningful Markdown cell or short sequence of Markdown cells.
2. Keep a **concept chunk** containing the Markdown explanation alone.
3. Create an **implementation chunk** that combines the explanation with the associated code.
4. Store links to both source cell positions so Cloud can cite the evidence accurately.

Example paired chunk:

```text
Notebook: preprocessing.ipynb
Topic: Leakage-safe preprocessing

Explanation:
Fit imputation and scaling on the training split only, otherwise information
from the test set influences the training process.

Implementation:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge())
])
```

**Checkpoint:** compare a few queries against the cell-only baseline. A conceptual question should retrieve a Markdown-rich chunk; an implementation question should retrieve the paired chunk.


In [8]:
concept_chunks = [] # this will store the markdown cell content as concept chunks
implementation_chunks = [] # this will store the code cell content as implementation chunks

# -- Step 1: Initilaizes 2 seperate lists for storing concept and implementation chunks respectively. --
# Grouping by notebook prevents a code cell from accidentally using Markdown
# from a different notebook when looking backward for context.
notebook_names = sorted({record["notebook"] for record in notebook_cells}) # get a sorted list of unique notebook names

for notebook_name in notebook_names: # iterate over each unique notebook name to go through each of its cells 
    records = [
        record for record in notebook_cells 
        if record["notebook"] == notebook_name # filter records to only include those from the current notebook
    ]
    records.sort(key=lambda record: record["cell_index"]) # sort the records by their cell index to maintain the original order within the notebook

    latest_markdown = None
    
    # right now records is a dictionary of all cells in the current notebook, sorted by their cell index. 
    # print example output
    print(f"Processing notebook: {notebook_name}")
    for record in records[:2]:  # print the first two records as an example
        print(record)

    for record in records:
        if record["cell_type"] == "markdown":
            latest_markdown = record

            concept_chunks.append(
                {
                    "content": record["content"],
                    "notebook": notebook_name,
                    "cell_index": record["cell_index"],
                    "cell_type": "markdown",
                    "chunk_type": "concept",
                }
            )
            continue

        if record["cell_type"] == "code":
            explanation = latest_markdown["content"] if latest_markdown else ""
            explanation_cell_index = (
                latest_markdown["cell_index"] if latest_markdown else None
            )
            paired_content = (
                f"Notebook: {notebook_name}\n"
                f"Explanation:\n{explanation}\n\n"
                f"Code:\n{record['content']}"
            )

            implementation_chunks.append(
                {
                    "content": paired_content,
                    "notebook": notebook_name,
                    "markdown_cell_index": explanation_cell_index,
                    "code_cell_index": record["cell_index"],
                    "cell_type": "markdown_and_code",
                    "chunk_type": "implementation",
                }
            )

print(f"Created {len(concept_chunks)} concept chunk(s).")
print(f"Created {len(implementation_chunks)} implementation chunk(s).")

# code ends up with two separate lists of chunks: `concept_chunks` and `implementation_chunks`, next cell we combine them.

print("\nExample implementation chunk:")
if implementation_chunks:
    print(implementation_chunks[0]["content"][:500])

Processing notebook: model_evaluation_error_analysis.ipynb
{'content': '# Model Evaluation & Error Analysis\n\n### *Evaluation is how we decide whether a model is useful for a real decision, not merely whether it produces a high score.*\n\nThis notebook continues the housing example from supervised learning. The same rows and features support two different model tasks:\n\n1. **Regression** predicts a number: a home\'s expected price.\n2. **Classification** predicts the probability of a category: whether a home is above a price boundary.\n\nThose outputs answer different questions, so they need different evaluation tools. A $100,000 price miss has a size; a classification mistake has a type, such as incorrectly flagging a standard home as premium. Neither family of metrics is universally better. The right choice depends on the output, the action someone will take, and the cost of each kind of error.\n\n### What You Will Practice\n\n- Build simple baselines that define what "useful" mean

## Phase 3: Embed and Retrieve the Evidence

Use a local embedding model such as `sentence-transformers/all-MiniLM-L6-v2`. Create an embedding for every chunk, embed a user question, and rank chunks with cosine similarity.

Begin with an in-memory list of chunk records and `scikit-learn` similarity search. For a small notebook collection, this is transparent, fast enough, and easy to debug.

Use a small value of $k$, usually $3$ to $5$, for the top retrieved chunks. Print the results before connecting an LLM.

**Checkpoint:** make a small set of retrieval questions, such as:

- "Why should preprocessing be fit on the training data only?"
- "What is calibration?"
- "How did the notebook encode categorical features?"
- "When is PR-AUC useful?"

For each question, inspect whether the expected explanation and/or related code appear in the top-$k$ chunks. Fix parsing or chunking first when retrieval misses the right evidence.

In [9]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Combine concept and implementation chunks into one searchable collection.
# this create a super dict combining both concept and implementation chunks
all_chunks = [
    {**chunk, "chunk_type": "concept"}
    for chunk in concept_chunks
] + [
    {**chunk, "chunk_type": "implementation"}
    for chunk in implementation_chunks
]

print(f"Total chunks: {len(all_chunks)}")
# this is just a quick summary of the number of chunks by type
print(
    "Concept chunks: "
    f"{sum(chunk['chunk_type'] == 'concept' for chunk in all_chunks)}"
)
print(
    "Implementation chunks: "
    f"{sum(chunk['chunk_type'] == 'implementation' for chunk in all_chunks)}"
)

if not all_chunks: 
    raise ValueError("No chunks available. Run the Phase 1 and Phase 2 cells first.")


# Phase 3 starts with a transparent lexical baseline. Each chunk uses its
# 'content' field, which is the schema created in Phase 2.
corpus = [chunk["content"] for chunk in all_chunks]
vectorizer = TfidfVectorizer(stop_words="english")
chunk_vectors = vectorizer.fit_transform(corpus)

print(f"Chunk vectors shape: {chunk_vectors.shape}")


def search_chunks(query, chunk_type_filter=None, top_n=3):
    """Return the highest-scoring chunks for a query.

    Args:
        query: Natural-language question to search for.
        chunk_type_filter: Optional 'concept' or 'implementation' filter.
        top_n: Maximum number of results to return.
    """
    valid_filters = {None, "concept", "implementation"}
    if chunk_type_filter not in valid_filters:
        raise ValueError("chunk_type_filter must be None, 'concept', or 'implementation'.")
    if top_n < 1:
        raise ValueError("top_n must be at least 1.")

    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, chunk_vectors).ravel()
    ranked_indices = np.argsort(similarities)[::-1]

    results = []
    for index in ranked_indices:
        record = all_chunks[index]

        if chunk_type_filter and record["chunk_type"] != chunk_type_filter:
            continue
        if similarities[index] <= 0:
            continue

        results.append(
            {
                "record": record,
                "similarity_score": round(float(similarities[index]), 4),
            }
        )

        if len(results) >= top_n:
            break

    return results


example_results = search_chunks("Why is preprocessing is a crucial step?")
print(f"Example search returned {len(example_results)} result(s).")
for result in example_results:
    record = result["record"]
    print(f"{result['similarity_score']} | {record['notebook']} | " f"{record['chunk_type']}")
    # print the actual content of each result to verify it matched the example query
    print(record["content"])

Total chunks: 93
Concept chunks: 52
Implementation chunks: 41
Chunk vectors shape: (93, 1927)
Example search returned 3 result(s).
0.4093 | preprocessing.ipynb | concept
### Information about preprocessing:
    - Preprocessing is a crucial step in the machine learning pipeline. It involves transforming raw data into a format that is suitable for modeling. Proper preprocessing can improve model performance and prevent issues such as data leakage. Common preprocessing steps include handling missing values, encoding categorical variables, normalizing numerical features, and splitting data into training and testing sets.
    - Ensuring that preprocessing steps are applied consistently to both training and testing data to avoid data leakage.
    - Common pitfalls in preprocessing include data leakage, inconsistent handling of missing values, and improper encoding of categorical variables.
    - It is important to document and justify each preprocessing step to ensure reproducibility and tra

## Phase 4: Support Clarifying Follow-Ups

A follow-up like "break that down more" needs two kinds of context:

- **Conversation history** tells Cloud what "that" refers to.
- **Retrieved notebook chunks** remain the factual evidence for the next answer.

Keep a small recent message window, such as the latest two to four user-and-assistant turns. Re-run retrieval for every follow-up, using the current question plus enough recent context to resolve references. The model must still answer from retrieved notebook evidence rather than relying on chat history as a factual source.

Prompt shape:

```text
Use retrieved notebook context as the factual source.
Use the recent conversation only to resolve follow-up wording.
Answer only from the retrieved evidence.
Cite the notebook source for factual claims.
State clearly when the notebook evidence is insufficient.
```


In [10]:
from collections import deque

TOPIC_KEYWORDS = {
    "preprocessing": {
        "preprocessing", "missing", "imputation", "impute", "encoding",
        "categorical", "scaling", "scale", "standardize", "leakage",
    },
    "model evaluation": {
        "evaluation", "metric", "metrics", "mae", "rmse", "r2", "roc",
        "auc", "pr-auc", "precision", "recall", "calibration", "threshold",
        "error", "errors",
    },
    "supervised learning": {
        "supervised", "regression", "classification", "ridge", "lasso", "logistic",
        "random forest", "decision tree", "prediction",
    },
    "unsupervised learning": {
        "clustering", "cluster", "k-means", "dbscan", "pca", "anomaly",
        "isolation forest", "unsupervised", "dimensionality",
    },
}


def classify_question(question):
    """Return an inspectable topic and question type for a user question."""
    normalized_question = question.lower()
    topic_scores = {
        topic: sum(keyword in normalized_question for keyword in keywords)
        for topic, keywords in TOPIC_KEYWORDS.items()
    }
    matched_topics = [
        topic for topic, score in topic_scores.items()
        if score > 0
    ]
    if {
        "supervised learning",
        "unsupervised learning",
    }.issubset(matched_topics):
        topic = "supervised vs unsupervised learning"
    elif matched_topics:
        topic = max(matched_topics, key=lambda item: topic_scores[item])
    else:
        topic = "general ML"

    if any(marker in normalized_question for marker in {"break down", "simpler", "clarify", "more"}):
        question_type = "clarification"
    elif any(marker in normalized_question for marker in {"compare", "versus", "difference", "better"}):
        question_type = "comparison"
    elif "how different" in normalized_question:
        question_type = "comparison"
    elif any(marker in normalized_question for marker in {"how", "show", "code", "implement"}):
        question_type = "implementation"
    elif any(marker in normalized_question for marker in {"why", "explain", "mean", "define", "what is"}):
        question_type = "concept explanation"
    else:
        question_type = "general question"

    return {
        "topic": topic,
        "question_type": question_type,
        "topic_scores": topic_scores,
    }


def previous_topic(conversation):
    """Find the most recent specific topic in the conversation."""
    for message in reversed(list(conversation)):
        topic = message.get("topic")
        if topic and topic != "general ML":
            return topic
    return "general ML"


def add_message(conversation, role, content, topic=None, question_type=None, max_messages=4):
    """Add a message with inspectable topic metadata."""
    if role not in {"user", "assistant"}:
        raise ValueError("role must be 'user' or 'assistant'.")
    if not content or not content.strip():
        raise ValueError("content must be a non-empty string.")

    classification = classify_question(content) if role == "user" else {}
    inferred_topic = classification.get("topic", "unknown")
    if role == "user" and inferred_topic == "general ML":
        inferred_topic = previous_topic(conversation)

    conversation.append(
        {
            "role": role,
            "content": content.strip(),
            "topic": topic or inferred_topic,
            "question_type": question_type or classification.get("question_type", "response"),
        }
    )
    while len(conversation) > max_messages:
        conversation.popleft()
    return conversation


def format_conversation(conversation):
    """Format recent messages with topic metadata for a future LLM prompt."""
    return "\n".join(
        f"Topic: {message.get('topic', 'unknown')} | "
        f"Type: {message.get('question_type', 'unknown')}\n"
        f"{message['role'].title()}: {message['content']}"
        for message in conversation
    )

# This helps query chat history so follow-up questions retain context.
def build_retrieval_query(question, conversation):
    """Add recent topic context to a follow-up before searching chunks."""
    classification = classify_question(question)
    topic = classification["topic"]
    if topic == "general ML":
        topic = previous_topic(conversation)

    recent_context = "\n".join(
        message["content"]
        for message in list(conversation)[-4:]
        if message["content"] != question
    )

    follow_up_markers = {
        "that", "it", "this", "those", "these", "more", "why", "how", "before", "previously", "above"
    }
    question_words = set(question.lower().replace("?", "").split())
    is_follow_up = bool(question_words & follow_up_markers) and bool(recent_context)

    if is_follow_up:
        return (
            f"Topic: {topic}\n"
            f"Previous conversation:\n{recent_context}\n\n"
            f"Follow-up question:\n{question}"
        )
    return question


def retrieve_with_conversation(question, conversation, top_n=3, chunk_type_filter=None):
    """Retrieve notebook evidence plus the question classification."""
    classification = classify_question(question)
    topic = classification["topic"]
    if topic == "general ML":
        topic = previous_topic(conversation)

    retrieval_query = build_retrieval_query(question, conversation)
    results = search_chunks(
        retrieval_query,
        chunk_type_filter=chunk_type_filter,
        top_n=top_n,
    )

    return {
        "question": question,
        "topic": topic,
        "question_type": classification["question_type"],
        "retrieval_query": retrieval_query,
        "sources": results,
    }


# Demonstrate a two-turn conversation without using an LLM yet.
conversation = deque(maxlen=4)
add_message(conversation, "user", "What is regularization?")
first_turn = retrieve_with_conversation(
    "What is regularization?",
    conversation,
    top_n=2,
)

if first_turn["sources"]:
    first_source = first_turn["sources"][0]["record"]["content"]
    add_message(
        conversation,
        "assistant",
        first_source,
        topic=first_turn["topic"],
        question_type="retrieved evidence",
    )

follow_up_question = "Can you break that down more for me?"
add_message(conversation, "user", follow_up_question)
follow_up = retrieve_with_conversation(
    follow_up_question,
    conversation,
    top_n=2,
)

print("Current question classification:")
print(f"Topic: {follow_up['topic']} | Type: {follow_up['question_type']}")
print("\nRecent conversation:")
print(format_conversation(conversation))
print("\nSearch query used for the follow-up:")
print(follow_up["retrieval_query"])
print("\nRetrieved follow-up sources:")
for result in follow_up["sources"]:
    record = result["record"]
    print(
        f"{result['similarity_score']} | {record['notebook']} | "
        f"{record['chunk_type']}"
    )

Current question classification:
Topic: general ML | Type: clarification

Recent conversation:
Topic: general ML | Type: concept explanation
User: What is regularization?
Topic: general ML | Type: clarification
User: Can you break that down more for me?

Search query used for the follow-up:
Topic: general ML
Previous conversation:
What is regularization?

Follow-up question:
Can you break that down more for me?

Retrieved follow-up sources:
0.0624 | supervised_learning.ipynb | concept
0.0612 | supervised_learning.ipynb | implementation


## Phase 5: Generate Grounded Answers

After retrieval is dependable, pass the user question and top-$k$ notebook chunks to a local Ollama model such as `qwen2.5:7b`. The model's job is to synthesize and explain the retrieved evidence, not to invent new notebook content.

The RAG engine should return an answer plus its cited sources to the Streamlit app:

```python
{
    "answer": "PR-AUC is useful here because...",
    "sources": [
        {
            "notebook": "model_evaluation_error_analysis.ipynb",
            "cell_index": 24,
            "cell_type": "markdown",
            "content": "...",
            "score": 0.91
        }
    ]
}
```

Move the tested ingestion and retrieval functions from this notebook into `rag_assistant/rag_engine.py`. `index_notebooks()` should parse, chunk, embed, and store the evidence; `query()` should retrieve evidence, create the prompt, and return the answer and sources.


In [11]:
import ollama

def build_grounded_prompt(question, topic, question_type, conversation, sources):
    """Build the prompt that gives the LLM instructions and notebook evidence."""
    formatted_sources = "\n\n".join(
        f"Source {index}: {result['record']['notebook']} "
        f"({result['record']['chunk_type']}, "
        f"similarity={result['similarity_score']})\n"
        f"{result['record']['content']}"
        for index, result in enumerate(sources, start=1)
    )

    return f"""You are Cloud, an assistant for the ML Foundations notebooks.
            Answer the user's question using only the retrieved notebook evidence below.
            Use the conversation only to understand follow-up references such as 'that' or 'it' (you can reference the follow_up_markers for more keywords to look out for).
            Explain the evidence clearly at the requested level.
            Do not invent facts that are not supported by the sources.
            If the sources are insufficient, say that clearly.
            Mention the source notebook when making factual claims.

            Detected topic: {topic}
            Question type: {question_type}

            Recent conversation:
            {format_conversation(conversation)}

            User question:{question}

            Retrieved notebook evidence: {formatted_sources}
        """


def generate_grounded_answer(question, conversation, top_n=3, model="qwen2.5:7b"): # why this model? bc it is designed to handle complex reasoning and context effectively. 
    """Retrieve notebook evidence and ask Ollama for a grounded answer."""
    retrieved = retrieve_with_conversation(
        question,
        conversation,
        top_n=top_n,
    )

    prompt = build_grounded_prompt(
        question=question,
        topic=retrieved["topic"],
        question_type=retrieved["question_type"],
        conversation=conversation,
        sources=retrieved["sources"],
    )

    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )

    return {
        "answer": response["message"]["content"],
        "topic": retrieved["topic"],
        "question_type": retrieved["question_type"],
        "sources": retrieved["sources"],
        "prompt": prompt,
    }


# Example: Phase 4 identifies the follow-up topic, Phase 5 generates a new answer.
phase_5_response = generate_grounded_answer(
    question=follow_up_question,
    conversation=conversation,
    top_n=2, # this limits the number of retrieved sources to 2 (past two conversation turns)
)

print(f"Topic: {phase_5_response['topic']}")
print(f"Question type: {phase_5_response['question_type']}")
print("\nCloud's grounded answer:\n")
print(phase_5_response["answer"])
print("\nSources used:")
for source in phase_5_response["sources"]:
    record = source["record"]
    print(
        f"- {record['notebook']} | {record['chunk_type']} | "
        f"score={source['similarity_score']}"
    )

Topic: general ML
Question type: clarification

Cloud's grounded answer:

The notebook evidence provided does not directly define or explain regularization. The content focuses on general concepts and workflows in supervised learning, including features, targets, and various machine learning models. Regularization is not explicitly mentioned in the given passages.

To clarify regularization, it is a technique used in machine learning to prevent overfitting by adding a penalty to the loss function during training. This penalty discourages the model from assigning too much importance to any single feature. Common types of regularization include L1 (Lasso) and L2 (Ridge) regularization.

If you would like a more detailed explanation of regularization, I would recommend looking into specific notebook sections that discuss model training and overfitting prevention techniques.

Sources used:
- supervised_learning.ipynb | concept | score=0.0624
- supervised_learning.ipynb | implementation | s


## Phase 6: Evaluate Before Adding Complexity

Create 15 to 25 questions that cover direct facts, concept explanations, code implementation, and questions the notebooks cannot answer. Record which source cell(s) should support each answer.

Evaluate:

- **Retrieval hit rate:** did the expected evidence appear in the top-$k$ chunks?
- **Groundedness:** does the answer stay within retrieved notebook content?
- **Citation correctness:** does each displayed source support the claim?
- **Latency:** how long do embedding, retrieval, and generation take?

Only after this baseline is working should you persist the vectors with ChromaDB or experiment with reranking. The learning value is in understanding why a chunk was retrieved, whether it is the right evidence, and how a change improves those measurements.

